In [105]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.offline import plot


In [109]:
state_filepath='/Users/melodyqian/Documents/GitHub/FindMyNuclearWaste/CSVs/State_Level_Demographics_filtered.csv'
statedf=pd.read_csv(state_filepath)
site_filepath='/Users/melodyqian/Documents/GitHub/FindMyNuclearWaste/CSVs/MasterDataset.csv'
sitedf=pd.read_csv(site_filepath)

In [110]:
sitedf= sitedf[sitedf['county_data']!=1]
sitedf=sitedf[sitedf['Type']!= 'Government Facility/Multiple']

In [125]:
sitelabelstor = sitedf['Storage'].value_counts().index
sitevaluestor = sitedf['Storage'].value_counts().values
sitelabelurb= sitedf['Urban_Rural'].value_counts().index
sitevalueurb= sitedf['Urban_Rural'].value_counts().values

colors1 = ['DarkKhaki', 'CadetBlue','Khaki']
storfig=go.Figure(data=go.Pie(
     values=sitevaluestor,
     labels=sitelabelstor,
     #domain=dict(x=[0, 0.5]),
     name="Storage Types",
     textposition='outside',
     marker=dict(colors=colors1, line=dict(color="#FFFFFF", width=2)),
     hole=.3))

storfig.update_layout(
    title_text = 'Type of Waste Storage',
    width=500
)
storfig.show()

In [124]:
colors2 = ['PaleGoldenRod', 'DarkSeaGreen','White', 'DarkGrey']
urbfig= go.Figure(data=go.Pie(
     values=sitevalueurb,
     labels=sitelabelurb,
     #domain=dict(x=[0.5, 1.0]),
     name="Urban/Rural",
     textposition='outside',
     marker=dict(colors=colors2, line=dict(color="#FFFFFF", width=2)),
     hole=.3))

urbfig.update_layout(
    title_text = 'Urban-Rural Classification of Waste Disposal Sites',
    width=500
)
urbfig.show()

In [84]:
statecolumns=['white_percent', 'black_percent', 'asian_percent', 'pacific_percent', 'native_percent', 'hispanic_percent', 'nonhispanic_percent']
sitecolumns=['white_percent', 'black_percent', 'asian_percent', 'pacific_percent', 'native_percent', 'hispanic_percent', 'nonhispanic_percent']

# Melt the data
meltedstate = statedf[statecolumns].melt(var_name='Demographic', value_name='Value')
mstate = meltedstate.groupby('Demographic')['Value'].mean().reset_index()

meltedsite = sitedf[sitecolumns].melt(var_name='Demographic', value_name='Value')
msite = meltedsite.groupby('Demographic')['Value'].mean().reset_index()

# Rename the demographics
rename_map = {
    'white_percent': 'White',
    'black_percent': 'Black',
    'asian_percent': 'Asian',
    'pacific_percent': 'Pacific Islander',
    'native_percent': 'Native',
    'hispanic_percent': 'Hispanic',
    'nonhispanic_percent': 'Non-Hispanic'
}

mstate['Demographic'] = mstate['Demographic'].replace(rename_map)
msite['Demographic'] = msite['Demographic'].replace(rename_map)

# Define the desired order
order = ['White', 'Black', 'Asian', 'Pacific Islander', 'Native', 'Hispanic', 'Non-Hispanic']

# Sort both dataframes by the desired order
mstate['Demographic'] = pd.Categorical(mstate['Demographic'], categories=order, ordered=True)
mstate = mstate.sort_values('Demographic').reset_index(drop=True)

msite['Demographic'] = pd.Categorical(msite['Demographic'], categories=order, ordered=True)
msite = msite.sort_values('Demographic').reset_index(drop=True)
#msitedf = sitedf[sitecolumns].mean().to_frame().T


In [96]:
msite['hover_text'] = msite.apply(
    lambda row: f"{row['Demographic']}: {round(row['Value'] * 100, 2)}%", axis=1
)
mstate['hover_text'] = mstate.apply(
    lambda row: f"{row['Demographic']}: {round(row['Value'] * 100, 2)}%", axis=1
)

In [97]:
print(msite)

        Demographic     Value              hover_text
0             White  0.895231           White: 89.52%
1             Black  0.060154            Black: 6.02%
2             Asian  0.010462            Asian: 1.05%
3  Pacific Islander  0.000000  Pacific Islander: 0.0%
4            Native  0.005231           Native: 0.52%
5          Hispanic  0.050923         Hispanic: 5.09%
6      Non-Hispanic  0.949692    Non-Hispanic: 94.97%


In [104]:
racetnfig= go.Figure()
racetnfig.add_trace(go.Bar(
    x=mstate['Demographic'],
    y=mstate['Value'],
    name='State',
    marker_color='rgb(129, 127, 0)',
    text=mstate['hover_text'],
    textposition='none'
))
racetnfig.add_trace(go.Bar(
    x=msite['Demographic'],
    y=msite['Value'],
    name='Site',
    text=msite['hover_text'],
    marker_color='rgb(211, 228, 36)',
    textposition='none'
))
 
racetnfig.update_layout(
    title_text = 'Racial and Ethnic Makeup Comparison',
    width=900
)

racetnfig.show()

In [6]:
comparison_df = pd.merge(
    statehousecollapsed[['state_abbrev', 'house_value_median']], 
    sitehousecollapsed[['state_abbrev', 'house_value_median']], 
    on='state_abbrev', 
    suffixes=('_state', '_site')
)

comparison_df['site_sub_state']=comparison_df['house_value_median_site']-comparison_df['house_value_median_state']
sorteddf = comparison_df.sort_values('site_sub_state')

In [ ]:
fig = go.Figure(data=go.Choropleth(
    locations=comparison_df['state_abbrev'], # Spatial coordinates
    z = comparison_df['site_sub_state'].astype(float), # Data to be color-coded
    locationmode = 'USA-states', # set of locations match entries in `locations`
    colorscale = 'RdYlGn',
    colorbar_title = "Difference ($)",
    colorbar=dict(
        x=0.2,
        xref="container",
    ),
    showscale=True,
    zmax=50000,
    zmin=-50000,
    zmid=0
))


scatter_trace  = go.Scattergeo(
        lon = sitedf['longitude'],
        lat = sitedf['latitude'],
        mode = 'markers',
        marker=dict(
            color='rgb(211, 228, 36)',
            size=4,
            line=dict(
                color='Black',
                width=.5
            )
        ),
        text=sitedf['Site']
        )
fig.add_trace(scatter_trace)
fig.update_layout(
    title_text = 'Median House Value around Storage Site compared to State Median',
    geo_scope='usa', # limite map scope to USA
    width=900
)

fig.write_html('HouseCloropleth.html')

In [81]:
state_income_collapsed = statedf.groupby('state_abbrev')['household_income_median'].median()
state_income_collapsed= state_income_collapsed.reset_index()
site_income_collapsed = sitedf.groupby('state_abbrev')['household_income_median'].median()
site_income_collapsed= site_income_collapsed.reset_index()

In [85]:
comparison_df = pd.merge(
    state_income_collapsed[['state_abbrev', 'household_income_median']], 
    site_income_collapsed[['state_abbrev', 'household_income_median']], 
    on='state_abbrev', 
    suffixes=('_state', '_site')
)

comparison_df['site_sub_state']=comparison_df['household_income_median_site']-comparison_df['household_income_median_state']
sorteddf = comparison_df.sort_values('site_sub_state')

In [161]:
bubblefig = go.Figure()


bubblefig.add_trace(go.Scatter(
    x=sorteddf['household_income_median_state'], 
    y=sorteddf['state_abbrev'],
    mode='markers',
    marker=dict(
        size=numpy.sqrt(sorteddf['site_sub_state'].abs())/3,
        color=sorteddf['site_sub_state'],  # color by the same value
        colorscale='RdYlGn',
        colorbar_title = "Median Household Income",
        cmax=50000,
        cmid=0,
        cmin=-50000,
        showscale=True
    ),
    text=f'Difference={sorteddf['site_sub_state']}<br>State={sorteddf['state_abbrev']}',  # optional: for hover info
    name='States'
))

In [77]:
fig = go.Figure(data=go.Choropleth(
    locations=comparison_df['state_abbrev'], # Spatial coordinates
    z = comparison_df['site_sub_state'].astype(float), # Data to be color-coded
    locationmode = 'USA-states', # set of locations match entries in `locations`
    colorscale = 'RdYlGn',
    colorbar_title = "Difference between Median Household Incomes",
    showscale=True,
    zmax=50000,
    zmin=-50000,
    zmid=0
))


scatter_trace  = go.Scattergeo(
        lon = sitedf['longitude'],
        lat = sitedf['latitude'],
        mode = 'markers',
        marker=dict(
            color='rgb(211, 228, 36)',
            size=4,
            line=dict(
                color='Black',
                width=.5
            )
        ),
        text=sitedf['Site']
        )
fig.add_trace(scatter_trace)
fig.update_layout(
    title_text = 'Median Household Income around Storage Site compared to State Median',
    geo_scope='usa', # limite map scope to USA
)

fig.show()